In [1]:
import numpy as np
import pandas as pd

ratings = pd.DataFrame({
    "Alice":  [5, 4, 0, 0, 0],
    "Bob":    [5, 5, 2, 1, 2],
    "Charlie": [4, 5, 1, 2, 1],
    "David":  [1, 1, 5, 5, 0],
    "Emily": [0, 2, 0 , 2 , 1]
}, index=["Avengers", "Iron Man", "Titanic", "The Notebook", "Batman"])

ratings

,Alice,Bob,Charlie,David,Emily
Avengers,5,5,4,1,0
Iron Man,4,5,5,1,2
Titanic,0,2,1,5,0
The Notebook,0,1,2,5,2
Batman,0,2,1,0,1


In [2]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(ratings)

sim_df = pd.DataFrame(
    similarity,
    index=ratings.index,
    columns=ratings.index
)

sim_df

,Avengers,Iron Man,Titanic,The Notebook,Batman
Avengers,1.000000,0.956924,0.423795,0.377134,0.698257
Iron Man,0.956924,1.000000,0.433351,0.488475,0.823653
Titanic,0.423795,0.433351,1.000000,0.908025,0.372678
The Notebook,0.377134,0.488475,0.908025,1.000000,0.420084
Batman,0.698257,0.823653,0.372678,0.420084,1.000000


In [3]:
#CF CLass exercise to recommend movie to ALice

alice_ratings = ratings["Alice"]

alice_ratings

Avengers        5
Iron Man        4
Titanic         0
The Notebook    0
Batman          0
Name: Alice, dtype: int64

In [4]:
Target_rating = "Titanic"

numerator = 0
denominator = 0

for item,  ratings  in alice_ratings.items():
    if ratings > 0:
        similarity = sim_df.loc[Target_rating, item]
    
        numerator += similarity * ratings
        denominator += similarity

pred =  numerator/ denominator

print("Predicted rating for Titanic for Alice", round(pred, 2))

Predicted rating for Titanic for Alice 4.49


In [5]:
Target_rating = "The Notebook"

numerator = 0
denominator = 0

for item,  ratings in alice_ratings.items():
    if ratings > 0:
        similarity = sim_df.loc[Target_rating, item]
    
        numerator += similarity * ratings
        denominator += similarity

pred =  numerator/ denominator

print("Predicted rating for The Notebook for Alice", round(pred, 2))

Predicted rating for The Notebook for Alice 4.44


In [6]:
Target_rating = "Batman"

numerator = 0
denominator = 0

for item,  ratings in alice_ratings.items():
    if ratings > 0:
        similarity = sim_df.loc[Target_rating, item]
    
        numerator += similarity * ratings
        denominator += similarity

pred =  numerator/ denominator

print("Predicted rating for Batman for Alice", round(pred, 2))

Predicted rating for Batman for Alice 4.46


In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# ---- 1. Sample data ----
ratings = pd.DataFrame({
    'Item A': [5, 4, 1, 1, 0],
    'Item B': [3, 0, 1, 0, 1],
    'Item C': [0, 0, 0, 0, 5],
    'Item D': [1, 1, 5, 4, 4],
}, index=['User1', 'User2', 'User3', 'User4', 'User5'])

In [2]:
print(ratings)

       Item A  Item B  Item C  Item D
User1       5       3       0       1
User2       4       0       0       1
User3       1       1       0       5
User4       1       0       0       4
User5       0       1       5       4


In [3]:
user_means = ratings.replace(0, np.nan).mean(axis=1)
ratings_centered = ratings.replace(0, np.nan).sub(user_means, axis=0).fillna(0)


item_sim = cosine_similarity(ratings_centered.T)
item_sim_df = pd.DataFrame(item_sim, index=ratings.columns, columns=ratings.columns)

print(item_sim_df.round(3))

        Item A  Item B  Item C  Item D
Item A   1.000   0.206   0.000  -0.938
Item B   0.206   1.000  -0.868  -0.475
Item C   0.000  -0.868   1.000   0.166
Item D  -0.938  -0.475   0.166   1.000


In [4]:
def predict_rating(user, item, ratings, item_sim_df, k=2):
    user_ratings = ratings.loc[user]
    rated_items = user_ratings[user_ratings > 0].index
    rated_items = [i for i in rated_items if i != item]

    sims = item_sim_df.loc[item, rated_items]
    sims = sims.sort_values(ascending=False).head(k)

    if sims.abs().sum() == 0:
        return user_ratings[user_ratings > 0].mean()

    numerator = sum(sims[i] * user_ratings[i] for i in sims.index)
    denominator = sum(abs(sims[i]) for i in sims.index)

    return numerator/ denominator

In [5]:
pred = predict_rating("User1", "Item C", ratings, item_sim_df, k=2)
print(f"Predicted rating for useer 1 on item c: {pred:.2f}")

Predicted rating for useer 1 on item c: 1.00


In [7]:
def recommend_items(user, ratings, item_sim_df, n=3, k=2):
    user_ratings = ratings.loc[user]
    unrated = user_ratings[user_ratings == 0].index
    preds = {item: predict_rating(user, item, ratings, item_sim_df, k) for item in unrated}

    return sorted(preds.items(), key=lambda x:x[1], reverse=True)[:n]

print(f"Top recommendation for user 1: {recommend_items('User1', ratings, item_sim_df,)}")

Top recommendation for user 1: [('Item C', 1.0)]
